## Setup

In [226]:
import os
import sys

# Add FleetPy to path if needed
fleetpy_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if fleetpy_path not in sys.path:
    sys.path.append(fleetpy_path)

# Import FleetPy modules
from src.misc.globals import *
import src.evaluation.tutorial_analysis as analysis
import src.misc.config as config
from src.misc.init_modules import load_simulation_environment
import pandas as pd

print("✅ FleetPy modules imported successfully!")

✅ FleetPy modules imported successfully!


In [260]:
def create_zone_demand():

    # load demand
    rq_file_dir = "../../data/demand/FleetPy_Manhattan/demand/Manhattan_2018/matched/Manhattan_2019_corrected"

    # Combined approach
    demand_dfs = []
    for x in range(1, 8):
        rq_file_name = f"2018-11-1{x}_sample_5_1.csv"
        abs_req_f = os.path.join(rq_file_dir, rq_file_name)
        df = pd.read_csv(abs_req_f, dtype={"start": int, "end": int})
        df['day'] = x  # Add day identifier
        demand_dfs.append(df)

    # node to zone conversion
    zone_network_dir = "../../data/demand/FleetPy_Manhattan/zones/Manhattan_corrected_12min_max/Manhattan_2019_corrected"
    node_zone_f = os.path.join(zone_network_dir, "node_zone_info.csv")
    node_zone_df = pd.read_csv(node_zone_f)
    
    # Set node_index as index before converting to dict
    node_zone_df.set_index('node_index', inplace=True)
    node_to_zone_dict = node_zone_df['zone_id'].to_dict()

    # dict structure: {(time_window, origin_zone, dest_zone): [day1_count, day2_count, ..., day7_count]}
    zone_demand_dict = {}

    # process each day separately
    for day_idx, df in enumerate(demand_dfs):
        for index, row in df.iterrows():
            time = row["rq_time"]
            # round time down to nearest 10 minute window
            time_window = (time // 600) * 600
            
            origin_node = row["start"]
            destination_node = row["end"]
            
            origin_zone = node_to_zone_dict.get(origin_node, -1)
            dest_zone = node_to_zone_dict.get(destination_node, -1)
            
            # skip if invalid zones or zones outside 0-3 range
            if origin_zone == -1 or dest_zone == -1:
                continue
            if origin_zone < 0 or origin_zone > 3:
                continue
            if dest_zone < 0 or dest_zone > 3:
                continue
            
            # create key
            key = (time_window, origin_zone, dest_zone)
            
            # initialize list with 7 zeros if key doesn't exist
            if key not in zone_demand_dict:
                zone_demand_dict[key] = [0] * 7
            
            # increment count for this day
            zone_demand_dict[key][day_idx] += 1
    
    # Maximum time across all days
    max_time = max(df['rq_time'].max() for df in demand_dfs)
    print(max_time)

    return zone_demand_dict



zone_demand = create_zone_demand()

86397


In [265]:
import pandas as pd

zone_demand = create_zone_demand()

# Convert dictionary to DataFrame
data_rows = []
for key, counts in zone_demand.items():
    time, origin, dest = key
    row = {
        'time': time,
        'origin_zone': origin,
        'dest_zone': dest,
        'counts': counts  # Save as list
    }
    data_rows.append(row)

df = pd.DataFrame(data_rows)

# Save to CSV
df.to_csv('../../cfr/project_data_nyc.csv', index=False)
print(f"Saved {len(df)} rows to project_data_nyc.csv")

86397
Saved 976 rows to project_data_nyc.csv


In [263]:
for key, counts in list(zone_demand.items())[:20]:
    time, origin, dest = key
    print(f"Time: {time}, {origin} -> {dest}, Day Counts: {counts}")

print(f"\nTotal entries: {len(zone_demand)}")

Time: 56400, 2 -> 2, Day Counts: [3, 1, 1, 0, 0, 0, 2]
Time: 62400, 0 -> 0, Day Counts: [1, 0, 1, 1, 2, 0, 0]
Time: 1200, 1 -> 0, Day Counts: [2, 1, 1, 0, 1, 1, 2]
Time: 44400, 1 -> 0, Day Counts: [1, 1, 2, 1, 0, 1, 2]
Time: 48600, 1 -> 1, Day Counts: [5, 1, 3, 11, 1, 2, 3]
Time: 31200, 0 -> 1, Day Counts: [1, 0, 2, 3, 1, 0, 1]
Time: 46200, 1 -> 1, Day Counts: [4, 4, 2, 2, 2, 0, 5]
Time: 57000, 1 -> 1, Day Counts: [6, 2, 1, 2, 4, 1, 2]
Time: 31200, 2 -> 2, Day Counts: [1, 0, 2, 3, 1, 0, 2]
Time: 41400, 2 -> 2, Day Counts: [1, 0, 1, 0, 0, 0, 2]
Time: 3600, 2 -> 1, Day Counts: [1, 0, 0, 0, 0, 0, 0]
Time: 58200, 1 -> 1, Day Counts: [8, 3, 0, 3, 2, 2, 1]
Time: 58200, 2 -> 2, Day Counts: [1, 0, 0, 1, 0, 0, 2]
Time: 81000, 0 -> 1, Day Counts: [1, 3, 2, 2, 0, 1, 1]
Time: 46200, 0 -> 1, Day Counts: [4, 2, 0, 2, 1, 0, 3]
Time: 85800, 1 -> 1, Day Counts: [1, 2, 0, 1, 0, 3, 5]
Time: 70200, 0 -> 1, Day Counts: [1, 1, 1, 1, 0, 2, 1]
Time: 60600, 1 -> 1, Day Counts: [2, 4, 6, 3, 2, 1, 3]
Time: 7200,

In [ ]:
import numpy as np


# create_zone_demand()

# state + demand -> next state

# state[i] is number of cars in zone i
state1 = np.array([0, 0, 0, 0])
state2 = np.array([2, 3, 4, 5])

print('total capacity', state1.sum() + state2.sum())

# action[i] is the change in count of cars in zone i
action1 = np.array([-10, 0, 0, 10])
action2 = np.array([-10, 0, 0, 10])

# demand[i] is the number of riders needing to be picked up in zone i
# get allocated automatically during state transition
demand = np.array([0, 0, 0, 1])

print('total demand', demand.sum())

finished_rides1 = np.array([1, 0, 0, 0])
finished_rides2 = np.array([1, 0, 0, 0])

adjacency_dict = {
    0: [1, 2, 3],
    1: [0, 2, 3],
    2: [0, 1, 3],
    3: [0, 1, 2]
}


def state_transition(s1_operator1, s1_operator2, action1, action2, demand, finished_rides1, finished_rides2):
    
    # when demand appears in zone, randomly assign cars from operators that have cars in that zone 
    # if there is still demand, randomly assign cars from adjacent zones
    # but how to do that efficiency with an array?

    # raise exception if sum of action is not 0
    if action1.sum() != 0 or action2.sum() != 0:
        raise ValueError(f"Actions must sum to 0. action1 sum: {action1.sum()}, action2 sum: {action2.sum()}")


    s1_operator1_inital = s1_operator1.copy()
    s1_operator2_inital = s1_operator2.copy()

    # DEMAND ALLOCATION
    def assign_demand_to_operator(demand, s1_operator1, s1_operator2, zone_index, source_zone_index):
        # if only one operator can handle demand in the same zone, assign to that operator
        if s1_operator1[source_zone_index] > 0 and s1_operator2[source_zone_index] == 0:
            demand[zone_index] -= 1
            s1_operator1[source_zone_index] -= 1
        elif s1_operator1[source_zone_index] == 0 and s1_operator2[source_zone_index] > 0:
            demand[zone_index] -= 1
            s1_operator2[source_zone_index] -= 1
        # if both operators can handle demand in that zone, coin flip to see who gets it
        elif s1_operator1[source_zone_index] > 0 and s1_operator2[source_zone_index] > 0:
            operator_demand_assignment = np.random.randint(0, 2)
            if operator_demand_assignment:
                demand[zone_index] -= 1
                s1_operator1[source_zone_index] -= 1
            else:
                demand[zone_index] -= 1
                s1_operator2[source_zone_index] -= 1

    # iterate through zones
    for zone_index in range(len(s1_operator1)):
        # keep assigning cars if there is unmet demand and there are still cars in the zone
        while demand[zone_index] > 0 and (s1_operator1[zone_index] > 0 or s1_operator2[zone_index] > 0):
            # if operator has capacity, assign demand to that operator, break ties randomly uniformly
            assign_demand_to_operator(demand, s1_operator1, s1_operator2, zone_index, zone_index)

    # if the while loop is still going, have to assign demand from other squares
    # iterate through zones
    for zone_index in range(len(s1_operator1)):
        # if unmet demand, and there are still rides anywhere in the network
        # print(s1_operator1[adj_zone_index].sum(), s1_operator2[adj_zone_index].sum())
        while demand[zone_index] > 0 and (s1_operator1.sum() > 0 or s1_operator2.sum() > 0):
            
            # find adjacent zones that have available cars from either operator
            available_adj_zones = [
                z for z in adjacency_dict[zone_index] 
                if s1_operator1[z] > 0 or s1_operator2[z] > 0
            ]
            
            if len(available_adj_zones) > 0:
                # randomly select adjacent zone with available cars to draw from
                adj_zone_index = np.random.choice(available_adj_zones)
                assign_demand_to_operator(demand, s1_operator1, s1_operator2, zone_index, adj_zone_index)
            else:
                # no adjacent zones with available cars, break to avoid infinite loop
                break

    print('operator 1', s1_operator1)
    print('operator 2', s1_operator2)
    print('repositioning', action1)

    # actual demand that can be satisfied
    reward_operator1 = np.sum(s1_operator1_inital - s1_operator1).item()
    reward_operator2 = np.sum(s1_operator2_inital - s1_operator2).item()


    pre_reposition_s1_operator1 = s1_operator1.copy()
    pre_reposition_s1_operator2 = s1_operator2.copy()

    # REPOSITIONING

    # action: operator orders a repositioning without knowing how the demand will be 
    # randomly take cars from repositioning to prevent negative states
    def adjust_action(state, action):
        adjusted_action = action.copy()
        
        for i in range(len(state)):
            # check if applying action would result in negative state
            while state[i] + adjusted_action[i] < 0:
                # Find zones with positive repositioning action
                available_zones = np.where(adjusted_action > 0)[0]
                
                if len(available_zones) > 0:
                    # Randomly pick a zone to steal from
                    steal_from = np.random.choice(available_zones)
                    adjusted_action[steal_from] -= 1
                    adjusted_action[i] += 1
                else:
                    # No available zones to steal from, break to avoid infinite loop
                    break
        
        return adjusted_action

    # Adjust actions before applying them
    action1 = adjust_action(s1_operator1, action1)
    action2 = adjust_action(s1_operator2, action2)

    # Now apply the adjusted actions
    s1_operator1 += action1
    s1_operator2 += action2

    print('operator 1 after', s1_operator1)
    print('operator 2 after', s1_operator2)


    print('actual repositioning', s1_operator1 - pre_reposition_s1_operator1, s1_operator2 - pre_reposition_s1_operator2)
    print('final demand', demand)

    # ADD BACK CARS WITH FINISHED RIDES
    s1_operator1 += finished_rides1
    s1_operator2 += finished_rides2



    return s1_operator1, s1_operator2, reward_operator1, reward_operator2, demand
    # print(s2)

state_transition(state1, state2, action1, action2, demand, finished_rides1, finished_rides2)
    





total capacity 14
total demand 1
operator 1 [0 0 0 0]
operator 2 [2 3 4 4]
repositioning [-10   0   0  10]
operator 1 after [0 0 0 0]
operator 2 after [0 3 4 6]
actual repositioning [0 0 0 0] [-2  0  0  2]
final demand [0 0 0 0]


(array([1, 0, 0, 0]), array([1, 3, 4, 6]), 0, 1, array([0, 0, 0, 0]))

In [ ]:
# Create a dictionary with the essential parameters using global variable names
config_params = {
    # Simulation environment
    # Use immediate decisions simulation environment
    G_SIM_ENV: 'ImmediateDecisionsSimulation',

    # No max decision time for immediate decisions
    G_AR_MAX_DEC_T: 0,
    # Basic request type for testing. Always accepts the operator's offer
    G_RQ_TYP1: 'BasicRequest',

    # Network and Demand
    G_NETWORK_NAME: 'example_network',             # Basic network for testing
    G_DEMAND_NAME: 'example_demand',               # Example demand pattern
    G_ZONE_SYSTEM_NAME: 'example_zones',           # Service area zones

    # Time Settings
    G_SIM_TIME_STEP: 60,                           # Update every 30 seconds
    G_SIM_START_TIME: 0,                           # Start at midnight
    G_SIM_END_TIME: 3600*8,                          # Run for 1 hour

    # Operational Settings
    G_NETWORK_TYPE: 'NetworkBasicWithStore',       # Basic network with store
    G_OP_MAX_WT: 300,                              # Max wait time of 5 minutes
    G_OP_MAX_DTF: 1.4,                             # Allow 40% detour
    # Constant boarding time of 30 seconds
    G_OP_CONST_BT: 30,
    G_NR_OPERATORS: 1,                             # Number of operators
    # Value of time function for vehicle routing control
    G_OP_VR_CTRL_F: 'func_key:distance_and_user_times_with_walk;vot:0.45',

    # Simulation Settings
    G_SLAVE_CPU: 1,                                # Use 1 CPU for slave processes
    "log_level": "info",                           # Set log level to INFO
    # Use a fixed random seed for reproducibility
    G_RANDOM_SEED: 0
}

# Convert to DataFrame for CSV format
config_df = pd.DataFrame(list(config_params.items()),
                         columns=['Parameter', 'Value'])

# Save constant config
config_dir = 'scenarios'
os.makedirs(config_dir, exist_ok=True)
constant_config_path = os.path.join(config_dir, 'custom_constant_config.csv')
config_df.to_csv(constant_config_path, index=False)

# Create a minimal scenario config (can override constant config values)
scenario_params = {
    G_STUDY_NAME: 'game_study',                    # Name of the study
    G_SCENARIO_NAME: 'example_pool_irsonly_sc_1',  # Scenario name
    G_RQ_FILE: "example_100.csv",                  # Example request file
    G_OP_FLEET: "default_vehtype:10",              # Size of the fleet
    # Operational module for pooling with IRS
    G_OP_MODULE: 'PoolingIRSOnly',
    G_OP_REPO_M: 'GameRepositioning', # Our algorithm
    G_OP_REPO_TH_DEF: '0;900',
    G_OP_REPO_TS: 300,
    G_RA_FC_TYPE: 'perfect',
    G_RA_FC_TR: 300

}
scenario_df = pd.DataFrame([scenario_params])
scenario_path = os.path.join(config_dir, 'scenario_custom_config.csv')
scenario_df.to_csv(scenario_path, index=False)

print("✅ Custom configuration files created!")
print("\n📝 Current configuration:")
for param, value in config_params.items():
    print(f"  - {param:.<30} {value}")

## Run Simulation

In [ ]:
# Set up paths to our custom config files
scs_path = os.path.join(os.getcwd(), 'scenarios')
constant_config_file = os.path.join(scs_path, 'custom_constant_config.csv')
scenario_file = os.path.join(scs_path, 'scenario_custom_config.csv')

# Read configuration files
constant_cfg = config.ConstantConfig(constant_config_file)
scenario_cfgs = config.ScenarioConfig(scenario_file)

# Combine configurations
scenario_cfg = constant_cfg + scenario_cfgs[0]

# Initialize simulation
sim = load_simulation_environment(scenario_cfg)

# Run the simulation
sim.run()

## Analysis

In [ ]:
# Define the results directory based on the scenario name
results_dir = os.path.join(os.getcwd(), 'results',
                           scenario_cfg[G_SCENARIO_NAME])

In [ ]:
# Get KPI summary
kpi_summary = analysis.analyze_kpis(results_dir)

# Display KPIs
print('📈 Simulation KPIs:')
print(kpi_summary.to_string(index=False))

In [ ]:
# Analyze user statistics
fig, wait_time_stats = analysis.analyze_user_stats(results_dir)
fig.show()

print('\n⏱️ Wait Time Statistics (minutes):')
print(wait_time_stats.to_string())